In [ ]:
import pynq
from doa_mts import *
# o1 = pynq.Overlay("/boot/doa_mts.bit", download=False)
o1 = doaMtsOverlay("doa_mts.bit", download=False)

pynq.Device.active_device.reset(o1.parser, o1.timestamp, o1.bitfile_name)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants
from scipy.io import savemat
from IPython.display import display, clear_output
from rfsoc4x2 import oled


win_len_power = 13
fc = 5.7e9
min_ang = -60 # min scanning angle
max_ang = 60 # max scanning angle
scan_res = 1 # scan resolution
scan_axis = list(np.arange(min_ang, max_ang+1, scan_res)) # angles axis
num_elements = 4
lbd = scipy.constants.c / fc
d = lbd/2

A = np.zeros((len(scan_axis), 4), dtype=np.cdouble)
for t in range(len(scan_axis)):
    A[t, :] = np.exp(-1j * 2 * np.pi * d * np.arange(num_elements) * np.sin(np.radians(scan_axis[t])) / lbd)

channels = 8 # 4 I + 4 Q
N = len(o1.adc_capture) // channels

delay = 7

offset = 2**win_len_power
print(offset)

## Latency calculations
ram_depth = offset
freq = 125e6
clk_tick = 1/freq
ps_bw = 128
write_read = 2
raw_bw = 128
prod_bw = 512

latency_raw = write_read * ram_depth * clk_tick * raw_bw/ps_bw
latency_prod = write_read * ram_depth * clk_tick * prod_bw/ps_bw
print(latency_raw)
print(latency_prod)

while True:
    # %store -r test
    # print(test)
    print(o1.dataStream)
    ## RAW
    tic_raw = time.time() 
    AlignedCaptureSamples = np.zeros((1,len(o1.adc_capture)),dtype=np.int16)
    o1.internal_capture(AlignedCaptureSamples)

    ShapedCaptureSamples = np.zeros((channels, N), dtype=np.int16)
    for i in range(channels):
        ShapedCaptureSamples[i] = AlignedCaptureSamples[0][i::channels]

    iqData = np.zeros((4, N), dtype=np.cdouble)
    idx = 0
    for i in range(0, channels, 2):
        iqData[idx] = np.conjugate(ShapedCaptureSamples[i] - 1j * ShapedCaptureSamples[i+1])
        idx = idx + 1
    ## RAW


    rawData = iqData
    rawData = rawData[:,delay:offset+delay]
    R = rawData@np.conjugate(rawData.T)
    print(len(rawData[0,:]))
    # MVDR
    Rx = R  # Data covariance matrix
    Rx_Inv = np.linalg.inv(Rx)  # Inverse of covariance matrix   

    # Assuming scan_axis is already defined as a numpy array
    yspec = np.zeros(len(scan_axis))

    # Calculate yspec
    # B = np.sum((A@Rx_Inv), axis=0)@np.conj(A.T)
    # B = np.zeros((len(scan_axis), 1), dtype=np.cdouble)
    B = np.sum((A@Rx_Inv), axis=0)@(A.T)
    yspec = 10 * np.log10(np.abs(1 / B))

    ang_raw = scan_axis[np.argmax(yspec)]
    toc_raw = time.time()

    power_old = (np.max(yspec))
    yspec_old = yspec


    # Plot data
    fig = plt.figure()
    plt.plot(scan_axis, yspec_old, color='r', label='raw')
    plt.xlabel('Scan Axis, degree')
    plt.ylabel('Y-Spec, dB')
    plt.title('MVDR DOA Spectrum from two sources')
    plt.legend()
    plt.grid(True)

    print("Window length: {:d} samples".format(offset))
    print("Resolution: {:.3f} deg".format(scan_res))
    print("RAW: {:.3f} deg".format(ang_raw))
    t_raw = toc_raw - tic_raw
    print('{:.5f} sec RAW'.format(t_raw+latency_raw))
    plt.show()
    clear_output(wait = True)

In [ ]:
o1.data_size